In [ ]:
import os
import yfinance as yf
import time
import smtplib
from email.message import EmailMessage
import pandas as pd
# ==========================================
# RISK MANAGEMENT PARAMETERS (DAY 24)
# ==========================================
STOP_LOSS_PCT = -0.01   # Cut losses if position drops 1%
TAKE_PROFIT_PCT = 0.02  # Secure profits if position rises 2%

# --- 1. EMAIL FUNCTION ---
def send_trade_notification(signal, price, pnl):
    msg = EmailMessage()
    msg.set_content(f'Update From The Bot:\nAction: {signal}\nPrice: {price:.2f}\nPnL: {pnl:.2f}')
    msg['Subject'] = f'Bot Alert: {signal}'
    msg['From'] = 'smtp.quant@gmail.com'
    msg['To'] = 'smtp.quant@gmail.com'
    
    with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp:
        # NOTE: Use your valid 16-letter App Password here
        smtp.login('smtp.quant@gmail.com', 'bjxq rsbe ocmn fobw')
        smtp.send_message(msg)

# --- 2. INITIALIZE MEMORY ---
last_buy_price = 0
last_signal = 'None'

if os.path.exists('trade_log.csv'):
    try:
        with open('trade_log.csv','r') as f:
            lines = f.readlines()
            if len(lines) > 1:
                last_line = lines[-1].split(',')
                last_signal = last_line[2].strip()
                if last_signal == 'BUY':
                    last_buy_price = float(last_line[1])
        print(f'🤖 Memory recovered. Last signal was: {last_signal}')
    except Exception as e:
        print(f'⚠️ Memory recovery failed: {e}. Starting fresh.')
print('🚀 Quantum Engine: Online (Risk Mgmt & RSI Filter Active)')

# --- 3. MAIN EXECUTION LOOP ---
while True:
    try:
        # Pull data (using 5d so the 190 SMA and 14 RSI have plenty of historical data)
        data = yf.download('RELIANCE.NS', period='5d', interval='1m', progress=False)
        
        # --- TECHNICAL INDICATORS ---
        # Dual SMAs
        data['SMA_45'] = data['Close']['RELIANCE.NS'].rolling(window=45).mean()
        data['SMA_190'] = data['Close']['RELIANCE.NS'].rolling(window=190).mean()
        
        # Day 25: RSI Calculation (14-Period)
        close_prices = data['Close']['RELIANCE.NS']
        data['Delta'] = close_prices.diff()
        data['Gain'] = data['Delta'].clip(lower=0)
        data['Loss'] = data['Delta'].clip(upper=0).abs()
        data['Avg_Gain'] = data['Gain'].rolling(window=14).mean()
        data['Avg_Loss'] = data['Loss'].rolling(window=14).mean()
        data['RS'] = data['Avg_Gain'] / data['Avg_Loss']
        data['RSI'] = 100 - (100 / (1 + data['RS']))
        
        # Extract the latest values
        latest = data.iloc[-1]
        sma_45_val = latest['SMA_45'].item()
        sma_190_val = latest['SMA_190'].item()
        current_price = latest['Close']['RELIANCE.NS'].item()
        current_rsi = latest['RSI'].item()

        # ==========================================
        # DAY 24: THE RISK TRIPWIRE
        # ==========================================
        if last_signal == 'BUY' and last_buy_price > 0:
            current_pnl_pct = (current_price - last_buy_price) / last_buy_price
            trigger_signal = None
            
            # Check if we hit our risk limits
            if current_pnl_pct <= STOP_LOSS_PCT:
                trigger_signal = 'STOP_LOSS'
            elif current_pnl_pct >= TAKE_PROFIT_PCT:
                trigger_signal = 'TAKE_PROFIT'
                
            # Execute emergency sell if tripped
            if trigger_signal:
                pnl = current_price - last_buy_price
                print(f'\n🚨 {trigger_signal} HIT! Force Selling at {current_price:.2f} | PnL: {current_pnl_pct:.2%}')
                
                with open('trade_log.csv', 'a') as f:
                    f.write(f'{time.ctime()},{current_price},{trigger_signal},{pnl}\n')
                
                try:
                    send_trade_notification(trigger_signal, current_price, pnl)
                    print('📧 Emergency Notification Sent')
                except Exception as e:
                    print(f'⚠️ Notification Failed: {e}')
                
                # Put the bot in a cooldown state so it doesn't immediately buy again
                last_signal = 'COOLDOWN'
                last_buy_price = 0
                time.sleep(60)
                continue # Skip the rest of the loop for this minute

        # ==========================================
        # DAY 25: RSI + SMA DECISION ENGINE
        # ==========================================
        # The bot will only BUY if the SMA crosses UP *AND* the RSI is not overbought (< 60)
        if sma_45_val > sma_190_val and current_rsi < 60:
            current_signal = 'BUY'
        else:
            current_signal = 'SELL'

        # --- 4. EXECUTION LOGIC ---
        # If we are in COOLDOWN, we wait until the market naturally crosses to SELL to reset
        if last_signal == 'COOLDOWN':
            if current_signal == 'SELL':
                print('\n🔄 Cooldown finished. Market trend shifted down. System reset.')
                last_signal = 'SELL'
            else:
                pnl_display = 0
                print(f'{time.ctime()} | {current_price:.2f} | Status: COOLDOWN (Waiting for trend reset)')
                
        # Normal SMA/RSI Logic
        elif current_signal != last_signal:
            pnl = 0
            if current_signal == 'SELL' and last_buy_price > 0:
                pnl = current_price - last_buy_price
                print(f'\n💰 Trade Closed: PnL {pnl:.2f}')
                last_buy_price = 0 # Reset memory
                
            if current_signal == 'BUY':
                last_buy_price = current_price
                print(f'\n📥 Entry: Buying at {current_price:.2f} (RSI: {current_rsi:.2f})')
                
            with open('trade_log.csv','a') as f:
                f.write(f'{time.ctime()},{current_price},{current_signal},{pnl}\n')
                
            try:
                send_trade_notification(current_signal, current_price, pnl)
                print('📧 Notification Sent')
            except Exception as e:
                print(f'⚠️ Notification Failed: {e}')
                
            last_signal = current_signal
            
        else:
            # Day 25 Upgraded Heartbeat: Now includes live RSI tracking
            pnl_display = current_price - last_buy_price if last_signal == 'BUY' else 0
            # formatting RSI to handle cases where it might be NaN on the very first few minutes of the data fetch
            rsi_display = f"{current_rsi:.2f}" if not pd.isna(current_rsi) else "N/A" 
            print(f'{time.ctime()} | Price: {current_price:.2f} | RSI: {rsi_display} | Signal: {current_signal} | Open PnL: {pnl_display:.2f}')
            
    except Exception as e:
        print(f'\n⚠️ System Alert: {e}')
        time.sleep(10)
        continue
        
    time.sleep(60)

🤖 Memory recovered. Last signal was: SELL
🚀 Quantum Engine: Online (Risk Mgmt & RSI Filter Active)
Fri May  8 12:05:24 2026 | Price: 1424.00 | RSI: 34.78 | Signal: SELL | Open PnL: 0.00
